In [1]:
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd
from scipy import stats
from pathlib import Path
import sys
import os
from datetime import datetime
from nba_api.stats.endpoints import leaguedashteamstats

project_root = Path.cwd().parent.parent
project_root_str = str(project_root)

if project_root_str not in sys.path:
    sys.path.insert(0, project_root_str)

os.chdir(project_root_str)

from src.utils.team_info import nameDict

pd.set_option('display.max_columns', None)

In [2]:
today = datetime.today().strftime('%Y%m%d')

def get_latest_file(pattern):
    files = list(Path('data/raw/player_lines').glob(pattern))
    return max(files, key=lambda f: f.stat().st_mtime) if files else None

us_file = get_latest_file('NBA_US_*.csv')
dfs_file = get_latest_file('NBA_DFS_*.csv')

if us_file is None:
    raise ValueError("No US file found")

if dfs_file is None:
    raise ValueError("No DFS file found")

us_df = pd.read_csv(us_file)
lines_dfs = pd.read_csv(dfs_file)

lines_us = us_df[us_df['CATEGORY'] == 'player_points'].copy()


print("US file:", us_file.name)
print("DFS file:", dfs_file.name)
print("DFS latest pull:", lines_dfs['DATA_PULLED_AT'].max())
print("US latest pull:", us_df['DATA_PULLED_AT'].max())

US file: NBA_US_20260320_092055.csv
DFS file: NBA_DFS_20260320_092137.csv
DFS latest pull: 2026-03-20 09:21:37
US latest pull: 2026-03-20 09:20:55


In [3]:
us_df

,BOOKMAKER,CATEGORY,NAME,OVER/UNDER,LINE,ODDS,COMMENCE_TIME,LAST_UPDATE,DATA_PULLED_AT
0,FanDuel,player_points,Danny Wolf,Over,10.5,-130,2026-03-20,2026-03-20T16:20:28Z,2026-03-20 09:20:55
1,FanDuel,player_points,Danny Wolf,Under,10.5,-102,2026-03-20,2026-03-20T16:20:28Z,2026-03-20 09:20:55
2,FanDuel,player_points,Karl-Anthony Towns,Over,19.5,-104,2026-03-20,2026-03-20T16:20:28Z,2026-03-20 09:20:55
3,FanDuel,player_points,Karl-Anthony Towns,Under,19.5,-128,2026-03-20,2026-03-20T16:20:28Z,2026-03-20 09:20:55
4,FanDuel,player_points,OG Anunoby,Over,17.5,-118,2026-03-20,2026-03-20T16:20:28Z,2026-03-20 09:20:55
...,...,...,...,...,...,...,...,...,...
7663,Bovada,player_rebounds_assists,Scottie Barnes,Under,10.5,145,2026-03-21,2026-03-20T16:19:42Z,2026-03-20 09:20:55
7664,Bovada,player_rebounds_assists,Scottie Barnes,Over,11.5,-125,2026-03-21,2026-03-20T16:19:42Z,2026-03-20 09:20:55
7665,Bovada,player_rebounds_assists,Scottie Barnes,Under,11.5,-105,2026-03-21,2026-03-20T16:19:42Z,2026-03-20 09:20:55
7666,Bovada,player_rebounds_assists,Scottie Barnes,Over,12.5,120,2026-03-21,2026-03-20T16:19:42Z,2026-03-20 09:20:55


In [4]:
def get_latest_file(pattern):
    files = list(Path('data/raw/team_lines').glob(pattern))
    return max(files, key=lambda f: f.stat().st_mtime) if files else None

file = get_latest_file('NBA_*.json')

if file is None:
    raise ValueError("No JSON file found")

# try normal load first
try:
    team_dds = pd.read_json(file)
except ValueError:
    # fallback for nested JSON
    import json
    with open(file) as f:
        data = json.load(f)
    team_dds = pd.json_normalize(data)

print("Loaded:", file.name)
team_dds.head()

Loaded: NBA_20260320_092137.json


,home_team,away_team,commence_time,bookmakers
0,Brooklyn Nets,New York Knicks,2026-03-20 23:40:00+00:00,"[{'bookmaker': 'FanDuel', 'last_updated': '202..."
1,Detroit Pistons,Golden State Warriors,2026-03-20 23:40:00+00:00,"[{'bookmaker': 'DraftKings', 'last_updated': '..."
2,Houston Rockets,Atlanta Hawks,2026-03-21 00:10:00+00:00,"[{'bookmaker': 'FanDuel', 'last_updated': '202..."
3,Memphis Grizzlies,Boston Celtics,2026-03-21 00:10:00+00:00,"[{'bookmaker': 'FanDuel', 'last_updated': '202..."
4,Minnesota Timberwolves,Portland Trail Blazers,2026-03-21 00:10:00+00:00,"[{'bookmaker': 'FanDuel', 'last_updated': '202..."


In [5]:
df = pd.read_csv('data/raw/season_stats/S26.csv').sort_values(by='GAME_DATE')
df.tail()

,SEASON_YEAR,PLAYER_ID,PLAYER_NAME,NICKNAME,TEAM_ID,TEAM_ABBREVIATION,TEAM_NAME,GAME_ID,GAME_DATE,MATCHUP,WL,MIN,FGM,FGA,FG_PCT,FG3M,FG3A,FG3_PCT,FTM,FTA,FT_PCT,OREB,DREB,REB,AST,TOV,STL,BLK,BLKA,PF,PFD,PTS,PLUS_MINUS,NBA_FANTASY_PTS,DD2,TD3,WNBA_FANTASY_PTS,AVAILABLE_FLAG,MIN_SEC,TEAM_COUNT,E_OFF_RATING,OFF_RATING,sp_work_OFF_RATING,E_DEF_RATING,DEF_RATING,sp_work_DEF_RATING,E_NET_RATING,NET_RATING,sp_work_NET_RATING,AST_PCT,AST_TO,AST_RATIO,OREB_PCT,DREB_PCT,REB_PCT,TM_TOV_PCT,E_TOV_PCT,EFG_PCT,TS_PCT,USG_PCT,E_USG_PCT,E_PACE,PACE,PACE_PER40,sp_work_PACE,PIE,POSS,FGM_PG,FGA_PG,TEAM_FGM,TEAM_FGA,TEAM_FG_PCT,TEAM_FG3M,TEAM_FG3A,TEAM_FG3_PCT,TEAM_FTM,TEAM_FTA,TEAM_FT_PCT,TEAM_OREB,TEAM_DREB,TEAM_REB,TEAM_AST,TEAM_TOV,TEAM_STL,TEAM_BLK,TEAM_BLKA,TEAM_PF,TEAM_PFD,TEAM_PTS,TEAM_PLUS_MINUS,TEAM_E_OFF_RATING,TEAM_OFF_RATING,TEAM_E_DEF_RATING,TEAM_DEF_RATING,TEAM_E_NET_RATING,TEAM_NET_RATING,TEAM_AST_PCT,TEAM_AST_TO,TEAM_AST_RATIO,TEAM_OREB_PCT,TEAM_DREB_PCT,TEAM_REB_PCT,TEAM_TM_TOV_PCT,TEAM_EFG_PCT,TEAM_TS_PCT,TEAM_E_PACE,TEAM_PACE,TEAM_PACE_PER40,TEAM_POSS,TEAM_PIE,OPP_TEAM_ID,OPP_OPP_ABBREVIATION_base,OPP_OPP_NAME_base,OPP_FGM,OPP_FGA,OPP_FG_PCT,OPP_FG3M,OPP_FG3A,OPP_FG3_PCT,OPP_FTM,OPP_FTA,OPP_FT_PCT,OPP_OREB,OPP_DREB,OPP_REB,OPP_AST,OPP_TOV,OPP_STL,OPP_BLK,OPP_BLKA,OPP_PF,OPP_PFD,OPP_PTS,OPP_PLUS_MINUS,OPP_E_OFF_RATING,OPP_OFF_RATING,OPP_E_DEF_RATING,OPP_DEF_RATING,OPP_E_NET_RATING,OPP_NET_RATING,OPP_AST_PCT,OPP_AST_TO,OPP_AST_RATIO,OPP_OREB_PCT,OPP_DREB_PCT,OPP_REB_PCT,OPP_TM_TOV_PCT,OPP_EFG_PCT,OPP_TS_PCT,OPP_E_PACE,OPP_PACE,OPP_PACE_PER40,OPP_POSS,OPP_PIE
108,2025-26,201572,Brook Lopez,Brook,1610612746,LAC,LA Clippers,22501012,2026-03-19T00:00:00,LAC @ NOP,L,28.835000,2,9,0.222,0,3,0.000,2,6,0.333,2,1,3,2,3,0,1,0,5,3,6,-8,12.6,0,0,13.0,1,28:50,1,106.0,105.2,105.2,124.4,119.0,119.0,-18.4,-13.8,-13.8,0.111,0.67,11.8,0.080,0.045,0.064,17.6,18.0,0.222,0.258,0.238,0.234,94.09,96.55,80.46,96.55,-0.079,58,2.0,9.0,35,76,0.461,10,32,0.313,19,32,0.594,9,30,39,20,16.0,11,2,2,21,19,99,-6.0,102.0,102.1,109.7,107.1,-7.8,-5.1,0.571,1.25,15.6,0.229,0.780,0.483,0.165,0.526,0.550,96.4,97.5,81.25,97,0.442,1610612746,LAC,LA Clippers,35,76,0.461,10,32,0.313,19,32,0.594,9,30,39,20,16.0,11,2,2,21,19,99,-6.0,102.0,102.1,109.7,107.1,-7.8,-5.1,0.571,1.25,15.6,0.229,0.780,0.483,0.165,0.526,0.550,96.4,97.5,81.25,97,0.442
107,2025-26,1642869,Noah Penda,Noah,1610612753,ORL,Orlando Magic,22501008,2026-03-19T00:00:00,ORL @ CHA,L,21.350000,3,8,0.375,2,6,0.333,0,0,0.000,2,0,2,2,1,0,0,0,0,0,8,0,12.4,0,0,14.0,1,21:21,1,103.0,112.2,112.2,118.4,112.2,112.2,-15.4,0.0,0.0,0.143,2.00,18.2,0.077,0.000,0.045,9.1,9.1,0.500,0.500,0.176,0.178,93.84,92.18,76.81,92.18,0.068,41,3.0,8.0,39,93,0.419,14,42,0.333,19,21,0.905,12,19,31,24,14.0,11,4,2,23,24,111,-19.0,106.5,114.4,125.1,131.3,-18.7,-16.9,0.615,1.71,17.0,0.375,0.575,0.458,0.144,0.495,0.543,104.1,98.0,81.67,97,0.406,1610612753,ORL,Orlando Magic,39,93,0.419,14,42,0.333,19,21,0.905,12,19,31,24,14.0,11,4,2,23,24,111,-19.0,106.5,114.4,125.1,131.3,-18.7,-16.9,0.615,1.71,17.0,0.375,0.575,0.458,0.144,0.495,0.543,104.1,98.0,81.67,97,0.406
106,2025-26,1631131,Oscar Tshiebwe,Oscar,1610612762,UTA,Utah Jazz,22501014,2026-03-19T00:00:00,UTA vs. MIL,W,15.466667,3,3,1.000,0,0,0.000,1,1,1.000,3,3,6,1,0,0,0,0,0,1,7,18,15.7,0,0,14.0,1,15:28,1,140.8,140.6,140.6,83.2,81.8,81.8,57.6,58.8,58.8,0.071,0.00,25.0,0.231,0.158,0.188,0.0,0.0,1.000,1.017,0.079,0.091,99.93,100.86,84.05,100.86,0.182,32,3.0,3.0,48,89,0.539,18,48,0.375,14,20,0.700,12,35,47,34,17.0,10,2,2,18,18,128,32.0,124.5,128.0,95.9,96.0,28.6,32.0,0.708,2.00,22.7,0.364,0.796,0.591,0.170,0.640,0.654,101.5,100.0,83.33,100,0.663,1610612762,UTA,Utah Jazz,48,89,0.539,18,48,0.375,14,20,0.700,12,35,47,34,17.0,10,2,2,18,18,128,32.0,124.5,128.0,95.9,96.0,28.6,32.0,0.708,2.00,22.7,0.364,0.796,0.591,0.170,0.640,0.654,101.5,100.0,83.33,100,0.663
114,2025-26,1642863,Khaman Maluach,Khaman,1610612756,PHX,Phoenix Suns,22501013,2026-03-19T00:00:00,PHX @ SAS,L,13.750000,2,3,0.667,0,0,0.000,0,0,0.000

In [6]:
# ── Parse game odds: consensus spread & total per team ────────────────────
game_rows = []

for game in team_dds.to_dict('records'):
    home = game['home_team']
    away = game['away_team']
    commence = game['commence_time']
    bookmakers = game['bookmakers']

    spreads_home, spreads_away, totals = [], [], []

    for bk in bookmakers:
        for market in bk['markets']:
            if market['market_key'] == 'spreads':
                for outcome in market['outcomes']:
                    if outcome['name'] == home:
                        spreads_home.append(outcome['point'])
                    elif outcome['name'] == away:
                        spreads_away.append(outcome['point'])
            elif market['market_key'] == 'totals':
                for outcome in market['outcomes']:
                    if outcome['name'] == 'Over':          # one side is enough
                        totals.append(outcome['point'])

    # Consensus = median across bookmakers; count = how many books reported
    consensus_total      = round(np.median(totals), 1)      if totals        else None
    consensus_spread_home = round(np.median(spreads_home), 1) if spreads_home else None
    consensus_spread_away = round(np.median(spreads_away), 1) if spreads_away else None
    n_books              = len(bookmakers)

    game_rows.append({
        'TEAM': home, 'OPPONENT': away,
        'TEAM_SPREAD': consensus_spread_home,
        'GAME_TOTAL':  consensus_total,
        'N_BOOKS':     n_books,
        'COMMENCE_TIME': commence,
        'HOME_AWAY': 'HOME',
    })
    game_rows.append({
        'TEAM': away, 'OPPONENT': home,
        'TEAM_SPREAD': consensus_spread_away,
        'GAME_TOTAL':  consensus_total,
        'N_BOOKS':     n_books,
        'COMMENCE_TIME': commence,
        'HOME_AWAY': 'AWAY',
    })

game_odds_df = pd.DataFrame(game_rows)

# ── Filter to players with Underdog lines ──────────────────────────────────
updated_names = []
for name in lines_dfs[(lines_dfs['BOOKMAKER'] == 'Underdog') & (lines_dfs['CATEGORY'] == 'player_points')]['NAME'].unique():
    updated_names.append(nameDict.get(name, name))

df = df[df['PLAYER_NAME'].isin(updated_names)].copy()

# ── Rolling stats (last 5 / last 10) ──────────────────────────────────────
df['AVG_MIN_L5']  = df.groupby('PLAYER_ID')['MIN'].transform(lambda x: x.rolling(5).mean().round(2))
df['STD_MIN_L5']  = df.groupby('PLAYER_ID')['MIN'].transform(lambda x: x.rolling(5).std().round(2))
df['AVG_PTS_L5']  = df.groupby('PLAYER_ID')['PTS'].transform(lambda x: x.rolling(5).mean().round(2))
df['STD_PTS_L5']  = df.groupby('PLAYER_ID')['PTS'].transform(lambda x: x.rolling(5).std().round(2))
df['MED_PTS_L5']  = df.groupby('PLAYER_ID')['PTS'].transform(lambda x: x.rolling(5).median().round(2))
df['STD_PTS_L10'] = df.groupby('PLAYER_ID')['PTS'].transform(lambda x: x.rolling(10).std().round(2))

df['MIN_CONSISTENCY'] = (df['AVG_MIN_L5'] / df['STD_MIN_L5']).round(2)

# ── Most recent row per player ─────────────────────────────────────────────
latest = df.groupby('PLAYER_ID').last().reset_index()

# ── Underdog lines ─────────────────────────────────────────────────────────
prop_pts = (
    lines_dfs[(lines_dfs['BOOKMAKER'] == 'PrizePicks') & (lines_dfs['CATEGORY'] == 'player_points')]
    [['NAME', 'LINE', 'ODDS', 'COMMENCE_TIME']]
    .rename(columns={'NAME': 'PLAYER_NAME'})
    .drop_duplicates('PLAYER_NAME')
)

merged = latest.merge(prop_pts, on='PLAYER_NAME', how='inner')

# ── Real book odds (best available per side) ───────────────────────────────
real_odds = lines_us[lines_us['CATEGORY'] == 'player_points'].rename(columns={'NAME': 'PLAYER_NAME'})

for side, col in [('Over', 'ODDS_OVER'), ('Under', 'ODDS_UNDER')]:
    best = (
        real_odds[real_odds['OVER/UNDER'] == side]
        .groupby(['PLAYER_NAME', 'LINE'])['ODDS'].max()
        .reset_index().rename(columns={'ODDS': col})
    )
    merged = merged.merge(best, on=['PLAYER_NAME', 'LINE'], how='left')

merged['ODDS_OVER']  = merged['ODDS_OVER'].fillna(-137).astype(int)
merged['ODDS_UNDER'] = merged['ODDS_UNDER'].fillna(-137).astype(int)

# ── Merge game-level spread & total ───────────────────────────────────────
# Assumes df / latest has a 'TEAM' column matching full team names in game_odds.json
merged = merged.merge(
    game_odds_df[['TEAM', 'OPPONENT', 'TEAM_SPREAD', 'GAME_TOTAL', 'N_BOOKS', 'HOME_AWAY']],
    left_on='TEAM_NAME',
    right_on='TEAM',
    how='left'
).drop(columns='TEAM')

# ── Implied probability from book odds ────────────────────────────────────
def implied_prob(american_odds):
    if american_odds > 0:
        return round(100 / (american_odds + 100), 3)
    else:
        return round(abs(american_odds) / (abs(american_odds) + 100), 3)

merged['IMP_PROB_OVER']  = merged['ODDS_OVER'].apply(implied_prob)
merged['IMP_PROB_UNDER'] = merged['ODDS_UNDER'].apply(implied_prob)

# ── Core metrics ───────────────────────────────────────────────────────────
merged['EDGE']     = (merged['AVG_PTS_L5'] - merged['LINE']).round(2)
merged['MED_EDGE'] = (merged['MED_PTS_L5'] - merged['LINE']).round(2)
merged['Z_SCORE']  = ((merged['LINE'] - merged['AVG_PTS_L5']) / merged['STD_PTS_L10']).round(3)

merged['PROB_OVER']  = (1 - stats.norm.cdf(merged['Z_SCORE'])).round(3)
merged['PROB_UNDER'] = stats.norm.cdf(merged['Z_SCORE']).round(3)

# ── Game-context features ──────────────────────────────────────────────────
# Pace/scoring environment: high total → more pts available league-wide that game
merged['TOTAL_BOOST'] = ((merged['GAME_TOTAL'] - 220) / 10).round(3)   # ~0 at league avg

# Spread proxy for role/usage: big favorite → star plays less 4Q, dog → may chase
# Positive spread = underdog (gets points), negative = favorite
merged['IS_UNDERDOG'] = (merged['TEAM_SPREAD'] > 0).astype(int)

# ── Cover rate (last 10) ───────────────────────────────────────────────────
cover_df = df.merge(merged[['PLAYER_NAME', 'LINE']], on='PLAYER_NAME', how='inner')

cover_windows = {'L5': 5, 'L10': 10, 'L15': 15}

cover = cover_df.groupby('PLAYER_NAME').apply(
    lambda g: pd.Series({
        'OVER_RATE_L5':  (g['PTS'].tail(5)  > g['LINE'].iloc[0]).mean().round(2),
        'OVER_RATE_L10': (g['PTS'].tail(10) > g['LINE'].iloc[0]).mean().round(2),
        'OVER_RATE_L15': (g['PTS'].tail(15) > g['LINE'].iloc[0]).mean().round(2),
        'OVER_RATE_SEASON': (g['PTS'] > g['LINE'].iloc[0]).mean().round(2),
    })
).reset_index()

merged = merged.merge(cover, on='PLAYER_NAME', how='left')

# ── EV % ───────────────────────────────────────────────────────────────────
def calc_ev(prob, american_odds):
    decimal = (american_odds / 100 + 1) if american_odds > 0 else (100 / abs(american_odds) + 1)
    return round(((prob * (decimal - 1)) - (1 - prob)) * 100, 2)

merged['EV_OVER']  = merged.apply(lambda r: calc_ev(r['PROB_OVER'],  r['ODDS_OVER']),  axis=1)
merged['EV_UNDER'] = merged.apply(lambda r: calc_ev(r['PROB_UNDER'], r['ODDS_UNDER']), axis=1)

# ── Filters + scoring ──────────────────────────────────────────────────────
merged = merged[(merged['AVG_MIN_L5'] >= 20) & (merged['STD_MIN_L5'] <= 8)]

merged['CONFIDENCE'] = (
    (merged['EDGE'] / merged['STD_PTS_L5'])
    + merged['OVER_RATE_L10']
    + merged['MIN_CONSISTENCY'] * 0.1
    + merged['TOTAL_BOOST'] * 0.15      # slight bump in high-scoring games
).round(2)

merged['BET_FLAG'] = (
    (merged['EDGE']           >  1.5) &
    (merged['OVER_RATE_L10'] >= 0.6) &
    (merged['STD_PTS_L5']     <  6.0) &
    (merged['PROB_OVER']      >= 0.60) &
    (merged['EV_OVER']        >  0)
)

# ── Output ─────────────────────────────────────────────────────────────────
output = merged[[
    'PLAYER_NAME', 'TEAM_NAME', 'OPPONENT', 'HOME_AWAY',  # ← TEAM_NAME here
    'TEAM_SPREAD', 'GAME_TOTAL',
    'LINE', 'ODDS_OVER', 'ODDS_UNDER',
    'IMP_PROB_OVER', 'IMP_PROB_UNDER',
    'AVG_PTS_L5', 'MED_PTS_L5', 'STD_PTS_L5','EDGE', 'MED_EDGE', 'Z_SCORE',
    'PROB_OVER', 'PROB_UNDER', 'EV_OVER', 'EV_UNDER',
    'OVER_RATE_L5', 'OVER_RATE_L10', 'OVER_RATE_L15', 'OVER_RATE_SEASON', 
    'AVG_MIN_L5', 'STD_MIN_L5',
    'MIN_CONSISTENCY', 'TOTAL_BOOST', 'IS_UNDERDOG',
    'CONFIDENCE', 'BET_FLAG', 'COMMENCE_TIME'
]].sort_values('EV_OVER', ascending=False)

tier1 = output[output['BET_FLAG']]

print(f"Total players: {len(output)}")
print(f"Tier 1 bets:   {len(tier1)}\n")
output.head(10)

Total players: 37
Tier 1 bets:   8



/var/folders/9q/5_554qsx5z70w9d_vkmvjg0h0000gn/T/ipykernel_64921/3087386537.py:134: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  cover = cover_df.groupby('PLAYER_NAME').apply(


,PLAYER_NAME,TEAM_NAME,OPPONENT,HOME_AWAY,TEAM_SPREAD,GAME_TOTAL,LINE,ODDS_OVER,ODDS_UNDER,IMP_PROB_OVER,IMP_PROB_UNDER,AVG_PTS_L5,MED_PTS_L5,STD_PTS_L5,EDGE,MED_EDGE,Z_SCORE,PROB_OVER,PROB_UNDER,EV_OVER,EV_UNDER,OVER_RATE_L5,OVER_RATE_L10,OVER_RATE_L15,OVER_RATE_SEASON,AVG_MIN_L5,STD_MIN_L5,MIN_CONSISTENCY,TOTAL_BOOST,IS_UNDERDOG,CONFIDENCE,BET_FLAG,COMMENCE_TIME
20,Nickeil Alexander-Walker,Atlanta Hawks,Houston Rockets,AWAY,3.5,227.0,18.5,-103,-114,0.507,0.533,26.0,22.0,9.35,7.5,3.5,-0.966,0.833,0.167,64.17,-68.65,0.8,0.6,0.53,0.59,34.86,2.78,12.54,0.70,1,2.76,False,2026-03-21
32,Amen Thompson,Houston Rockets,Atlanta Hawks,HOME,-3.5,227.0,18.5,-115,-108,0.535,0.519,21.4,23.0,3.91,2.9,4.5,-0.876,0.809,0.191,51.25,-63.21,0.8,0.8,0.60,0.47,38.35,4.70,8.16,0.70,0,2.46,True,2026-03-21
5,Karl-Anthony Towns,New York Knicks,Brooklyn Nets,AWAY,-17.5,214.5,19.5,-104,-115,0.510,0.535,24.0,22.0,6.78,4.5,2.5,-0.717,0.763,0.237,49.67,-55.69,0.8,0.5,0.60,0.52,32.40,2.31,14.03,-0.55,0,2.48,False,2026-03-20
28,Pat Spencer,Golden State Warriors,Detroit Pistons,AWAY,5.0,217.0,7.5,-114,-105,0.533,0.512,11.4,10.0,4.04,3.9,2.5,-0.826,0.796,0.204,49.42,-60.17,0.8,0.7,0.60,0.43,22.80,4.38,5.21,-0.30,1,2.14,True,2026-03-20
9,Gary Payton II,Golden State Warriors,Detroit Pistons,AWAY,5.0,217.0,9.5,-137,-137,0.578,0.578,14.4,14.0,2.88,4.9,4.5,-1.049,0.853,0.147,47.56,-74.57,1.0,0.8,0.80,0.31,22.56,3.36,6.71,-0.30,1,3.13,True,2026-03-20
29,Dyson Daniels,Atlanta Hawks,Houston Rockets,AWAY,3.5,227.0,11.5,-105,-111,0.512,0.526,14.2,15.0,3.96,2.7,3.5,-0.657,0.744,0.256,45.26,-51.34,0.8,0.7,0.60,0.46,33.45,2.95,11.34,0.70,1,2.62,True,2026-03-21
35,Olivier-Maxence Prosper,Memphis Grizzlies,Boston Celtics,HOME,15.0,229.0,10.5,-106,-107,0.515,0.517,13.8,13.0,4.15,3.3,2.5,-0.648,0.742,0.258,44.20,-50.09,0.8,0.8,0.73,0.36,23.27,2.24,10.39,0.90,1,2.77,True,2026-03-21
10,OG Anunoby,New York Knicks,Brooklyn Nets,AWAY,-17.5,214.5,17.5,-105,-110,0.512,0.524,21.8,22.0,4.71,4.3,4.5,-0.605,0.727,0.273,41.94,-47.88,0.8,0.5,0.47,0.50,32.19,3.53,9.12,-0.55,0,2.24,False,2026-03-20
31,Taylor Hendricks,Memphis Grizzlies,Boston Celtics,HOME,15.0,229.0,9.5,-110,-110,0.524,0.524,12.2,13.0,4.55,2.7,3.5,-0.609,0.729,0.271,39.17,-48.26,0.6,0.7,0.67,0.30,24.90,3.01,8.27,0.90,1,2.26,True,2026-03-21
21,Ty Jerome,Memphis Grizzlies,Boston Celtics,HOME,15.0,229.0,18.5,-114,-107,0.533,0.517,21.0,21.0,4.42,2.5,2.5,-0.623,0.733,0.267,37.60,-48.35,0.8,0.7,0.77,0.77,24.41,0.45,54.24,0.90,1,6.82,True,2026-03-21


In [7]:
# Generalized best bets for all PrizePicks categories
# (Over/Under best odds from US books + EV using rolling stat distribution)

prop_categories = [
    'player_points',
    'player_rebounds',
    'player_assists',
    'player_threes',
    'player_blocks',
    'player_steals',
    'player_points_rebounds_assists',
    'player_points_rebounds',
    'player_points_assists',
    'player_rebounds_assists',
]

cat_to_stat_cols = {
    'player_points': ['PTS'],
    'player_rebounds': ['REB'],
    'player_assists': ['AST'],
    'player_threes': ['FG3M'],
    'player_blocks': ['BLK'],
    'player_steals': ['STL'],
    'player_points_rebounds_assists': ['PTS', 'REB', 'AST'],
    'player_points_rebounds': ['PTS', 'REB'],
    'player_points_assists': ['PTS', 'AST'],
    'player_rebounds_assists': ['REB', 'AST'],
}


def implied_prob(american_odds: float) -> float:
    if american_odds > 0:
        return round(100 / (american_odds + 100), 3)
    return round(abs(american_odds) / (abs(american_odds) + 100), 3)


def calc_ev(prob: float, american_odds: float) -> float:
    decimal = (american_odds / 100 + 1) if american_odds > 0 else (100 / abs(american_odds) + 1)
    return round(((prob * (decimal - 1)) - (1 - prob)) * 100, 2)


# --- Parse game odds: consensus spread & total per team ---
if 'game_odds_df' not in globals():
    game_rows = []

    for game in team_dds.to_dict('records'):
        home = game['home_team']
        away = game['away_team']
        commence = game['commence_time']
        bookmakers = game['bookmakers']

        spreads_home, spreads_away, totals = [], [], []

        for bk in bookmakers:
            for market in bk['markets']:
                if market['market_key'] == 'spreads':
                    for outcome in market['outcomes']:
                        if outcome['name'] == home:
                            spreads_home.append(outcome['point'])
                        elif outcome['name'] == away:
                            spreads_away.append(outcome['point'])
                elif market['market_key'] == 'totals':
                    for outcome in market['outcomes']:
                        if outcome['name'] == 'Over':  # one side is enough
                            totals.append(outcome['point'])

        # Consensus = median across bookmakers; count = how many books reported
        consensus_total = round(np.median(totals), 1) if totals else None
        consensus_spread_home = round(np.median(spreads_home), 1) if spreads_home else None
        consensus_spread_away = round(np.median(spreads_away), 1) if spreads_away else None
        n_books = len(bookmakers)

        game_rows.append({
            'TEAM': home,
            'OPPONENT': away,
            'TEAM_SPREAD': consensus_spread_home,
            'GAME_TOTAL': consensus_total,
            'N_BOOKS': n_books,
            'COMMENCE_TIME': commence,
            'HOME_AWAY': 'HOME',
        })
        game_rows.append({
            'TEAM': away,
            'OPPONENT': home,
            'TEAM_SPREAD': consensus_spread_away,
            'GAME_TOTAL': consensus_total,
            'N_BOOKS': n_books,
            'COMMENCE_TIME': commence,
            'HOME_AWAY': 'AWAY',
        })

    game_odds_df = pd.DataFrame(game_rows)


# Base player log features
base_df = pd.read_csv('data/raw/season_stats/S26.csv').sort_values(by='GAME_DATE')

outputs = []

for category in prop_categories:
    stat_cols = cat_to_stat_cols[category]

    # --- Filter to players with Underdog lines for THIS category ---
    updated_names = []
    cat_underdog_names = lines_dfs[
        (lines_dfs['BOOKMAKER'] == 'Underdog') & (lines_dfs['CATEGORY'] == category)
    ]['NAME'].unique()

    for name in cat_underdog_names:
        updated_names.append(nameDict.get(name, name))

    df = base_df[base_df['PLAYER_NAME'].isin(updated_names)].copy()

    if df.empty:
        continue

    # Build the stat we are analyzing for this category (single stat or sum for combos)
    if len(stat_cols) == 1:
        df['STAT_VALUE'] = df[stat_cols[0]]
    else:
        df['STAT_VALUE'] = df[stat_cols].sum(axis=1)

    # --- Rolling stats (last 5 / last 10) for THIS stat ---
    df['AVG_MIN_L5'] = df.groupby('PLAYER_ID')['MIN'].transform(lambda x: x.rolling(5).mean().round(2))
    df['STD_MIN_L5'] = df.groupby('PLAYER_ID')['MIN'].transform(lambda x: x.rolling(5).std().round(2))

    df['AVG_STAT_L5'] = df.groupby('PLAYER_ID')['STAT_VALUE'].transform(lambda x: x.rolling(5).mean().round(2))
    df['STD_STAT_L5'] = df.groupby('PLAYER_ID')['STAT_VALUE'].transform(lambda x: x.rolling(5).std().round(2))
    df['MED_STAT_L5'] = df.groupby('PLAYER_ID')['STAT_VALUE'].transform(lambda x: x.rolling(5).median().round(2))
    df['STD_STAT_L10'] = df.groupby('PLAYER_ID')['STAT_VALUE'].transform(lambda x: x.rolling(10).std().round(2))

    df['MIN_CONSISTENCY'] = (df['AVG_MIN_L5'] / df['STD_MIN_L5']).round(2)

    # --- Most recent row per player ---
    latest = df.groupby('PLAYER_ID').last().reset_index()

    # --- PrizePicks lines for THIS category ---
    prop_lines = (
        lines_dfs[(lines_dfs['BOOKMAKER'] == 'PrizePicks') & (lines_dfs['CATEGORY'] == category)]
        [['NAME', 'LINE', 'ODDS', 'COMMENCE_TIME']]
        .rename(columns={'NAME': 'PLAYER_NAME'})
        .copy()
    )

    if prop_lines.empty:
        continue

    prop_lines['LINE'] = prop_lines['LINE'].astype(float)

    # Keep one line per player for this category (consistent with your existing logic)
    prop_lines = prop_lines.drop_duplicates('PLAYER_NAME')

    merged = latest.merge(prop_lines, on='PLAYER_NAME', how='inner')
    merged['CATEGORY'] = category

    # --- Real book odds (best available per side) ---
    real_odds = us_df[us_df['CATEGORY'] == category].rename(columns={'NAME': 'PLAYER_NAME'}).copy()

    # Ensure numeric odds/lines
    real_odds['LINE'] = real_odds['LINE'].astype(float)

    for side, col in [('Over', 'ODDS_OVER'), ('Under', 'ODDS_UNDER')]:
        best = (
            real_odds[real_odds['OVER/UNDER'] == side]
            .groupby(['PLAYER_NAME', 'LINE'])['ODDS'].max()
            .reset_index()
            .rename(columns={'ODDS': col})
        )
        merged = merged.merge(best, on=['PLAYER_NAME', 'LINE'], how='left')

    merged['ODDS_OVER'] = merged['ODDS_OVER'].fillna(-137).astype(int)
    merged['ODDS_UNDER'] = merged['ODDS_UNDER'].fillna(-137).astype(int)

    # --- Merge game-level spread & total ---
    merged = merged.merge(
        game_odds_df[['TEAM', 'OPPONENT', 'TEAM_SPREAD', 'GAME_TOTAL', 'N_BOOKS', 'HOME_AWAY']],
        left_on='TEAM_NAME',
        right_on='TEAM',
        how='left'
    ).drop(columns='TEAM')

    # --- Implied probability from book odds ---
    merged['IMP_PROB_OVER'] = merged['ODDS_OVER'].apply(implied_prob)
    merged['IMP_PROB_UNDER'] = merged['ODDS_UNDER'].apply(implied_prob)

    # --- Core metrics ---
    merged['EDGE'] = (merged['AVG_STAT_L5'] - merged['LINE']).round(2)
    merged['MED_EDGE'] = (merged['MED_STAT_L5'] - merged['LINE']).round(2)
    merged['Z_SCORE'] = ((merged['LINE'] - merged['AVG_STAT_L5']) / merged['STD_STAT_L10']).round(3)

    merged['PROB_OVER'] = (1 - stats.norm.cdf(merged['Z_SCORE'])).round(3)
    merged['PROB_UNDER'] = stats.norm.cdf(merged['Z_SCORE']).round(3)

    # --- Game-context features ---
    merged['TOTAL_BOOST'] = ((merged['GAME_TOTAL'] - 220) / 10).round(3)  # ~0 at league avg
    merged['IS_UNDERDOG'] = (merged['TEAM_SPREAD'] > 0).astype(int)

    # --- Cover rate (last 10-ish windows) ---
    cover_df = df.merge(merged[['PLAYER_NAME', 'LINE']], on='PLAYER_NAME', how='inner')

    cover_windows = {'L5': 5, 'L10': 10, 'L15': 15}

    cover = cover_df.groupby('PLAYER_NAME').apply(
        lambda g: pd.Series({
            'OVER_RATE_L5': (g['STAT_VALUE'].tail(5) > g['LINE'].iloc[0]).mean().round(2),
            'OVER_RATE_L10': (g['STAT_VALUE'].tail(10) > g['LINE'].iloc[0]).mean().round(2),
            'OVER_RATE_L15': (g['STAT_VALUE'].tail(15) > g['LINE'].iloc[0]).mean().round(2),
            'OVER_RATE_SEASON': (g['STAT_VALUE'] > g['LINE'].iloc[0]).mean().round(2),
        })
    ).reset_index()

    merged = merged.merge(cover, on='PLAYER_NAME', how='left')

    # --- EV % ---
    merged['EV_OVER'] = merged.apply(lambda r: calc_ev(r['PROB_OVER'], r['ODDS_OVER']), axis=1)
    merged['EV_UNDER'] = merged.apply(lambda r: calc_ev(r['PROB_UNDER'], r['ODDS_UNDER']), axis=1)

    # --- Filters + scoring ---
    merged = merged[(merged['AVG_MIN_L5'] >= 20) & (merged['STD_MIN_L5'] <= 8)].copy()

    merged['CONFIDENCE'] = (
        (merged['EDGE'] / merged['STD_STAT_L5'])
        + merged['OVER_RATE_L10']
        + merged['MIN_CONSISTENCY'] * 0.1
        + merged['TOTAL_BOOST'] * 0.15
    ).round(2)

    # Simple, stat-agnostic bet rule: value by EV and probability
    merged['BET_FLAG'] = (
        (merged['PROB_OVER'] >= 0.60)
        & (merged['EV_OVER'] > 0)
    )

    output_cat = merged[[
        'PLAYER_NAME',
        'TEAM_NAME',
        'OPPONENT',
        'HOME_AWAY',
        'TEAM_SPREAD',
        'GAME_TOTAL',
        'CATEGORY',
        'LINE',
        'ODDS_OVER',
        'ODDS_UNDER',
        'IMP_PROB_OVER',
        'IMP_PROB_UNDER',
        'AVG_STAT_L5',
        'MED_STAT_L5',
        'STD_STAT_L5',
        'EDGE',
        'MED_EDGE',
        'Z_SCORE',
        'PROB_OVER',
        'PROB_UNDER',
        'EV_OVER',
        'EV_UNDER',
        'OVER_RATE_L5',
        'OVER_RATE_L10',
        'OVER_RATE_L15',
        'OVER_RATE_SEASON',
        'AVG_MIN_L5',
        'STD_MIN_L5',
        'MIN_CONSISTENCY',
        'TOTAL_BOOST',
        'IS_UNDERDOG',
        'CONFIDENCE',
        'BET_FLAG',
        'COMMENCE_TIME',
    ]].sort_values('EV_OVER', ascending=False)

    outputs.append(output_cat)


output_all = pd.concat(outputs, ignore_index=True) if outputs else pd.DataFrame()

tier1_all = output_all[output_all['BET_FLAG']] if not output_all.empty else output_all

print('Total bets across categories:', len(output_all))
print('Tier 1 bets:', len(tier1_all))

if not output_all.empty:
    display(output_all.head(20))
    display(tier1_all.head(20))


Total bets across categories: 173
Tier 1 bets: 62


/var/folders/9q/5_554qsx5z70w9d_vkmvjg0h0000gn/T/ipykernel_64921/4159200528.py:203: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  cover = cover_df.groupby('PLAYER_NAME').apply(
/var/folders/9q/5_554qsx5z70w9d_vkmvjg0h0000gn/T/ipykernel_64921/4159200528.py:203: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  cover = cover_df.groupby('PLAYER_NAME').apply(
/var/folders/9q/5_554qsx5z70w9d_vkmvjg0h0000gn/T/ipykernel_64921/4

,PLAYER_NAME,TEAM_NAME,OPPONENT,HOME_AWAY,TEAM_SPREAD,GAME_TOTAL,CATEGORY,LINE,ODDS_OVER,ODDS_UNDER,IMP_PROB_OVER,IMP_PROB_UNDER,AVG_STAT_L5,MED_STAT_L5,STD_STAT_L5,EDGE,MED_EDGE,Z_SCORE,PROB_OVER,PROB_UNDER,EV_OVER,EV_UNDER,OVER_RATE_L5,OVER_RATE_L10,OVER_RATE_L15,OVER_RATE_SEASON,AVG_MIN_L5,STD_MIN_L5,MIN_CONSISTENCY,TOTAL_BOOST,IS_UNDERDOG,CONFIDENCE,BET_FLAG,COMMENCE_TIME
0,Nickeil Alexander-Walker,Atlanta Hawks,Houston Rockets,AWAY,3.5,227.0,player_points,18.5,-103,-114,0.507,0.533,26.0,22.0,9.35,7.5,3.5,-0.966,0.833,0.167,64.17,-68.65,0.8,0.6,0.53,0.59,34.86,2.78,12.54,0.70,1,2.76,True,2026-03-21
1,Amen Thompson,Houston Rockets,Atlanta Hawks,HOME,-3.5,227.0,player_points,18.5,-115,-108,0.535,0.519,21.4,23.0,3.91,2.9,4.5,-0.876,0.809,0.191,51.25,-63.21,0.8,0.8,0.60,0.47,38.35,4.70,8.16,0.70,0,2.46,True,2026-03-21
2,Karl-Anthony Towns,New York Knicks,Brooklyn Nets,AWAY,-17.5,214.5,player_points,19.5,-104,-115,0.510,0.535,24.0,22.0,6.78,4.5,2.5,-0.717,0.763,0.237,49.67,-55.69,0.8,0.5,0.60,0.52,32.40,2.31,14.03,-0.55,0,2.48,True,2026-03-20
3,Pat Spencer,Golden State Warriors,Detroit Pistons,AWAY,5.0,217.0,player_points,7.5,-114,-105,0.533,0.512,11.4,10.0,4.04,3.9,2.5,-0.826,0.796,0.204,49.42,-60.17,0.8,0.7,0.60,0.43,22.80,4.38,5.21,-0.30,1,2.14,True,2026-03-20
4,Gary Payton II,Golden State Warriors,Detroit Pistons,AWAY,5.0,217.0,player_points,9.5,-137,-137,0.578,0.578,14.4,14.0,2.88,4.9,4.5,-1.049,0.853,0.147,47.56,-74.57,1.0,0.8,0.80,0.31,22.56,3.36,6.71,-0.30,1,3.13,True,2026-03-20
5,Dyson Daniels,Atlanta Hawks,Houston Rockets,AWAY,3.5,227.0,player_points,11.5,-105,-111,0.512,0.526,14.2,15.0,3.96,2.7,3.5,-0.657,0.744,0.256,45.26,-51.34,0.8,0.7,0.60,0.46,33.45,2.95,11.34,0.70,1,2.62,True,2026-03-21
6,Olivier-Maxence Prosper,Memphis Grizzlies,Boston Celtics,HOME,15.0,229.0,player_points,10.5,-106,-107,0.515,0.517,13.8,13.0,4.15,3.3,2.5,-0.648,0.742,0.258,44.20,-50.09,0.8,0.8,0.73,0.36,23.27,2.24,10.39,0.90,1,2.77,True,2026-03-21
7,OG Anunoby,New York Knicks,Brooklyn Nets,AWAY,-17.5,214.5,player_points,17.5,-105,-110,0.512,0.524,21.8,22.0,4.71,4.3,4.5,-0.605,0.727,0.273,41.94,-47.88,0.8,0.5,0.47,0.50,32.19,3.53,9.12,-0.55,0,2.24,True,2026-03-20
8,Taylor Hendricks,Memphis Grizzlies,Boston Celtics,HOME,15.0,229.0,player_points,9.5,-110,-110,0.524,0.524,12.2,13.0,4.55,2.7,3.5,-0.609,0.729,0.271,39.17,-48.26,0.6,0.7,0.67,0.30,24.90,3.01,8.27,0.90,1,2.26,True,2026-03-21
9,Ty Jerome,Memphis Grizzlies,Boston Celtics,HOME,15.0,229.0,player_points,18.5,-114,-107,0.533,0.517,21.0,21.0,4.42,2.5,2.5,-0.623,0.733,0.267,37.60,-48.35,0.8,0.7,0.77,0.77,24.41,0.45,54.24,0.90,1,6.82,True,2026-03-21


,PLAYER_NAME,TEAM_NAME,OPPONENT,HOME_AWAY,TEAM_SPREAD,GAME_TOTAL,CATEGORY,LINE,ODDS_OVER,ODDS_UNDER,IMP_PROB_OVER,IMP_PROB_UNDER,AVG_STAT_L5,MED_STAT_L5,STD_STAT_L5,EDGE,MED_EDGE,Z_SCORE,PROB_OVER,PROB_UNDER,EV_OVER,EV_UNDER,OVER_RATE_L5,OVER_RATE_L10,OVER_RATE_L15,OVER_RATE_SEASON,AVG_MIN_L5,STD_MIN_L5,MIN_CONSISTENCY,TOTAL_BOOST,IS_UNDERDOG,CONFIDENCE,BET_FLAG,COMMENCE_TIME
0,Nickeil Alexander-Walker,Atlanta Hawks,Houston Rockets,AWAY,3.5,227.0,player_points,18.5,-103,-114,0.507,0.533,26.0,22.0,9.35,7.5,3.5,-0.966,0.833,0.167,64.17,-68.65,0.8,0.6,0.53,0.59,34.86,2.78,12.54,0.70,1,2.76,True,2026-03-21
1,Amen Thompson,Houston Rockets,Atlanta Hawks,HOME,-3.5,227.0,player_points,18.5,-115,-108,0.535,0.519,21.4,23.0,3.91,2.9,4.5,-0.876,0.809,0.191,51.25,-63.21,0.8,0.8,0.60,0.47,38.35,4.70,8.16,0.70,0,2.46,True,2026-03-21
2,Karl-Anthony Towns,New York Knicks,Brooklyn Nets,AWAY,-17.5,214.5,player_points,19.5,-104,-115,0.510,0.535,24.0,22.0,6.78,4.5,2.5,-0.717,0.763,0.237,49.67,-55.69,0.8,0.5,0.60,0.52,32.40,2.31,14.03,-0.55,0,2.48,True,2026-03-20
3,Pat Spencer,Golden State Warriors,Detroit Pistons,AWAY,5.0,217.0,player_points,7.5,-114,-105,0.533,0.512,11.4,10.0,4.04,3.9,2.5,-0.826,0.796,0.204,49.42,-60.17,0.8,0.7,0.60,0.43,22.80,4.38,5.21,-0.30,1,2.14,True,2026-03-20
4,Gary Payton II,Golden State Warriors,Detroit Pistons,AWAY,5.0,217.0,player_points,9.5,-137,-137,0.578,0.578,14.4,14.0,2.88,4.9,4.5,-1.049,0.853,0.147,47.56,-74.57,1.0,0.8,0.80,0.31,22.56,3.36,6.71,-0.30,1,3.13,True,2026-03-20
5,Dyson Daniels,Atlanta Hawks,Houston Rockets,AWAY,3.5,227.0,player_points,11.5,-105,-111,0.512,0.526,14.2,15.0,3.96,2.7,3.5,-0.657,0.744,0.256,45.26,-51.34,0.8,0.7,0.60,0.46,33.45,2.95,11.34,0.70,1,2.62,True,2026-03-21
6,Olivier-Maxence Prosper,Memphis Grizzlies,Boston Celtics,HOME,15.0,229.0,player_points,10.5,-106,-107,0.515,0.517,13.8,13.0,4.15,3.3,2.5,-0.648,0.742,0.258,44.20,-50.09,0.8,0.8,0.73,0.36,23.27,2.24,10.39,0.90,1,2.77,True,2026-03-21
7,OG Anunoby,New York Knicks,Brooklyn Nets,AWAY,-17.5,214.5,player_points,17.5,-105,-110,0.512,0.524,21.8,22.0,4.71,4.3,4.5,-0.605,0.727,0.273,41.94,-47.88,0.8,0.5,0.47,0.50,32.19,3.53,9.12,-0.55,0,2.24,True,2026-03-20
8,Taylor Hendricks,Memphis Grizzlies,Boston Celtics,HOME,15.0,229.0,player_points,9.5,-110,-110,0.524,0.524,12.2,13.0,4.55,2.7,3.5,-0.609,0.729,0.271,39.17,-48.26,0.6,0.7,0.67,0.30,24.90,3.01,8.27,0.90,1,2.26,True,2026-03-21
9,Ty Jerome,Memphis Grizzlies,Boston Celtics,HOME,15.0,229.0,player_points,18.5,-114,-107,0.533,0.517,21.0,21.0,4.42,2.5,2.5,-0.623,0.733,0.267,37.60,-48.35,0.8,0.7,0.77,0.77,24.41,0.45,54.24,0.90,1,6.82,True,2026-03-21


In [8]:
league_df  = leaguedashteamstats.LeagueDashTeamStats(
    league_id_nullable='00',
    per_mode_detailed='PerGame',
    measure_type_detailed_defense='Advanced'
).get_data_frames()[0]
team_stats = league_df.set_index('TEAM_ID')

opp_stats = (
    league_df[['TEAM_NAME', 'DEF_RATING', 'DEF_RATING_RANK', 'PACE', 'PACE_RANK']]
    .copy()
    .rename(columns={
        'TEAM_NAME':  'OPPONENT',      
        'DEF_RATING': 'OPP_DEF_RATING',
        'DEF_RATING_RANK': 'OPP_RANK_DEF_RATING',
        'PACE': 'OPP_PACE',
        'PACE_RANK' : 'OPP_PACE_RANK'}))

final = output_all.merge(opp_stats, on='OPPONENT', how='left')
final = final[[
    'PLAYER_NAME', 'LINE','CATEGORY','OPPONENT',
    'TEAM_SPREAD', 'GAME_TOTAL', 'OPP_DEF_RATING',	'OPP_RANK_DEF_RATING',	'OPP_PACE',	'OPP_PACE_RANK',
    'ODDS_OVER', 'ODDS_UNDER',
    'IMP_PROB_OVER', 'IMP_PROB_UNDER',
    'AVG_STAT_L5', 'MED_STAT_L5', 'STD_STAT_L5','EDGE', 'MED_EDGE', 'Z_SCORE',
    'PROB_OVER', 'PROB_UNDER', 'EV_OVER', 'EV_UNDER',
    'OVER_RATE_L5', 'OVER_RATE_L10', 'OVER_RATE_L15', 'OVER_RATE_SEASON', 
    'AVG_MIN_L5', 'STD_MIN_L5','MIN_CONSISTENCY', 'IS_UNDERDOG']].sort_values('EV_OVER', ascending=False)
final.head()

,PLAYER_NAME,LINE,CATEGORY,OPPONENT,TEAM_SPREAD,GAME_TOTAL,OPP_DEF_RATING,OPP_RANK_DEF_RATING,OPP_PACE,OPP_PACE_RANK,ODDS_OVER,ODDS_UNDER,IMP_PROB_OVER,IMP_PROB_UNDER,AVG_STAT_L5,MED_STAT_L5,STD_STAT_L5,EDGE,MED_EDGE,Z_SCORE,PROB_OVER,PROB_UNDER,EV_OVER,EV_UNDER,OVER_RATE_L5,OVER_RATE_L10,OVER_RATE_L15,OVER_RATE_SEASON,AVG_MIN_L5,STD_MIN_L5,MIN_CONSISTENCY,IS_UNDERDOG
67,Nickeil Alexander-Walker,1.5,player_steals,Houston Rockets,3.5,227.0,112.4,8,96.68,29,-106,-119,0.515,0.543,2.4,2.0,0.55,0.9,0.5,-1.216,0.888,0.112,72.57,-79.39,1.0,0.7,0.60,0.41,34.86,2.78,12.54,1
69,Dyson Daniels,23.5,player_points_rebounds_assists,Houston Rockets,3.5,227.0,112.4,8,96.68,29,100,-115,0.500,0.535,27.4,29.0,4.83,3.9,5.5,-1.010,0.844,0.156,68.80,-70.83,0.8,0.8,0.67,0.55,33.45,2.95,11.34,1
0,Nickeil Alexander-Walker,18.5,player_points,Houston Rockets,3.5,227.0,112.4,8,96.68,29,-103,-114,0.507,0.533,26.0,22.0,9.35,7.5,3.5,-0.966,0.833,0.167,64.17,-68.65,0.8,0.6,0.53,0.59,34.86,2.78,12.54,1
70,Jalen Brunson,35.5,player_points_rebounds_assists,Brooklyn Nets,-17.5,214.5,118.2,27,97.33,27,-110,-114,0.524,0.533,40.0,40.0,2.12,4.5,4.5,-1.016,0.845,0.155,61.32,-70.90,1.0,0.7,0.53,0.59,37.11,3.41,10.88,0
71,Gary Payton II,16.5,player_points_rebounds_assists,Detroit Pistons,5.0,217.0,109.0,2,100.05,19,-120,-110,0.545,0.524,23.6,24.0,4.34,7.1,7.5,-1.101,0.865,0.135,58.58,-74.23,1.0,0.7,0.67,0.27,22.56,3.36,6.71,1


In [9]:
rename_map = {
    'PLAYER_NAME': 'Player',
    'CATEGORY': 'Prop',
    'LINE': 'Line',
    'OPPONENT': 'Opponent',
    'TEAM_SPREAD': 'Spread',
    'GAME_TOTAL': 'Total',
    'OPP_DEF_RATING': 'Opp Def Rating',
    'OPP_RANK_DEF_RATING': 'Opp Def Rank',
    'OPP_PACE': 'Opp Pace',
    'OPP_PACE_RANK': 'Opp Pace Rank',
    'ODDS_OVER': 'Odds Over',
    'ODDS_UNDER': 'Odds Under',
    'IMP_PROB_OVER': 'Implied Over',
    'IMP_PROB_UNDER': 'Implied Under',
    'AVG_STAT_L5': 'Avg Stat L5',
    'MED_STAT_L5': 'Med Stat L5',
    'STD_STAT_L5': 'Std Stat L5',
    'EDGE': 'Edge',
    'MED_EDGE': 'Med Edge',
    'Z_SCORE': 'Z Score',
    'PROB_OVER': 'Prob Over',
    'PROB_UNDER': 'Prob Under',
    'EV_OVER': 'EV Over',
    'EV_UNDER': 'EV Under',
    'OVER_RATE_L5': 'OVER L5',
    'OVER_RATE_L10': 'OVER L10',
    'OVER_RATE_L15': 'OVER L15',
    'OVER_RATE_SEASON': 'ALL SEASON',
    'AVG_MIN_L5': 'Avg Min L5',
    'STD_MIN_L5': 'Std Min L5',
    'MIN_CONSISTENCY': 'Min Consistency',
    'IS_UNDERDOG': 'Underdog'
}

df = final.rename(columns=rename_map)
df.head()

,Player,Line,Prop,Opponent,Spread,Total,Opp Def Rating,Opp Def Rank,Opp Pace,Opp Pace Rank,Odds Over,Odds Under,Implied Over,Implied Under,Avg Stat L5,Med Stat L5,Std Stat L5,Edge,Med Edge,Z Score,Prob Over,Prob Under,EV Over,EV Under,OVER L5,OVER L10,OVER L15,ALL SEASON,Avg Min L5,Std Min L5,Min Consistency,Underdog
67,Nickeil Alexander-Walker,1.5,player_steals,Houston Rockets,3.5,227.0,112.4,8,96.68,29,-106,-119,0.515,0.543,2.4,2.0,0.55,0.9,0.5,-1.216,0.888,0.112,72.57,-79.39,1.0,0.7,0.60,0.41,34.86,2.78,12.54,1
69,Dyson Daniels,23.5,player_points_rebounds_assists,Houston Rockets,3.5,227.0,112.4,8,96.68,29,100,-115,0.500,0.535,27.4,29.0,4.83,3.9,5.5,-1.010,0.844,0.156,68.80,-70.83,0.8,0.8,0.67,0.55,33.45,2.95,11.34,1
0,Nickeil Alexander-Walker,18.5,player_points,Houston Rockets,3.5,227.0,112.4,8,96.68,29,-103,-114,0.507,0.533,26.0,22.0,9.35,7.5,3.5,-0.966,0.833,0.167,64.17,-68.65,0.8,0.6,0.53,0.59,34.86,2.78,12.54,1
70,Jalen Brunson,35.5,player_points_rebounds_assists,Brooklyn Nets,-17.5,214.5,118.2,27,97.33,27,-110,-114,0.524,0.533,40.0,40.0,2.12,4.5,4.5,-1.016,0.845,0.155,61.32,-70.90,1.0,0.7,0.53,0.59,37.11,3.41,10.88,0
71,Gary Payton II,16.5,player_points_rebounds_assists,Detroit Pistons,5.0,217.0,109.0,2,100.05,19,-120,-110,0.545,0.524,23.6,24.0,4.34,7.1,7.5,-1.101,0.865,0.135,58.58,-74.23,1.0,0.7,0.67,0.27,22.56,3.36,6.71,1


In [10]:
prop_label_map = {
    'player_points': 'PTS',
    'player_rebounds': 'REB',
    'player_assists': 'AST',
    'player_assist': 'AST',  # just in case
    'player_threes': '3PM',
    'player_blocks': 'BLK',
    'player_steals': 'STL',
    'player_points_rebounds_assists': 'PTS+REB+AST',
    'player_points_rebounds': 'PTS+REB',
    'player_points_assists': 'PTS+AST',
    'player_rebounds_assists': 'REB+AST',
}

# In this cell `df['Prop']` is the CATEGORY value (e.g., 'player_points').
df['Prop'] = df['Prop'].map(prop_label_map).fillna(df['Prop'])

df = df[[
    'Player',
    'Prop',
    'Line',
    'Opponent',
    'Odds Over',
    'Odds Under',
    'Implied Over',
    'Implied Under',
    'EV Over',
    'EV Under',
    'Avg Stat L5',
    'Std Stat L5',
    'Prob Over',
    'Prob Under',
    'OVER L5',
    'OVER L10',
    'OVER L15',
    'Avg Min L5',
    'Spread',
    'Total',
    'Opp Def Rating',
    'Opp Def Rank',
    'Opp Pace',
    'Opp Pace Rank',
]].sort_values(by='EV Over', ascending=False)
df.head(10)

,Player,Prop,Line,Opponent,Odds Over,Odds Under,Implied Over,Implied Under,EV Over,EV Under,Avg Stat L5,Std Stat L5,Prob Over,Prob Under,OVER L5,OVER L10,OVER L15,Avg Min L5,Spread,Total,Opp Def Rating,Opp Def Rank,Opp Pace,Opp Pace Rank
67,Nickeil Alexander-Walker,STL,1.5,Houston Rockets,-106,-119,0.515,0.543,72.57,-79.39,2.4,0.55,0.888,0.112,1.0,0.7,0.60,34.86,3.5,227.0,112.4,8,96.68,29
69,Dyson Daniels,PTS+REB+AST,23.5,Houston Rockets,100,-115,0.500,0.535,68.80,-70.83,27.4,4.83,0.844,0.156,0.8,0.8,0.67,33.45,3.5,227.0,112.4,8,96.68,29
0,Nickeil Alexander-Walker,PTS,18.5,Houston Rockets,-103,-114,0.507,0.533,64.17,-68.65,26.0,9.35,0.833,0.167,0.8,0.6,0.53,34.86,3.5,227.0,112.4,8,96.68,29
70,Jalen Brunson,PTS+REB+AST,35.5,Brooklyn Nets,-110,-114,0.524,0.533,61.32,-70.90,40.0,2.12,0.845,0.155,1.0,0.7,0.53,37.11,-17.5,214.5,118.2,27,97.33,27
71,Gary Payton II,PTS+REB+AST,16.5,Detroit Pistons,-120,-110,0.545,0.524,58.58,-74.23,23.6,4.34,0.865,0.135,1.0,0.7,0.67,22.56,5.0,217.0,109.0,2,100.05,19
72,Donovan Clingan,PTS+REB+AST,25.5,Minnesota Timberwolves,-104,-118,0.510,0.541,57.32,-63.42,32.8,7.56,0.802,0.198,0.8,0.6,0.67,27.70,2.5,231.5,112.8,9,101.55,9
117,Nickeil Alexander-Walker,PTS+REB,21.5,Houston Rockets,-104,-109,0.510,0.522,57.12,-61.84,29.0,11.38,0.801,0.199,0.8,0.6,0.53,34.86,3.5,227.0,112.4,8,96.68,29
73,Cameron Johnson,PTS+REB+AST,17.5,Toronto Raptors,-112,-105,0.528,0.512,56.92,-66.61,23.8,3.35,0.829,0.171,1.0,0.7,0.60,31.05,-7.0,238.5,112.0,7,99.25,22
74,Christian Braun,PTS+REB+AST,19.5,Toronto Raptors,-115,-105,0.535,0.512,56.11,-67.79,26.4,8.29,0.835,0.165,0.8,0.7,0.67,32.98,-7.0,238.5,112.0,7,99.25,22
1,Amen Thompson,PTS,18.5,Atlanta Hawks,-115,-108,0.535,0.519,51.25,-63.21,21.4,3.91,0.809,0.191,0.8,0.8,0.60,38.35,-3.5,227.0,113.1,11,102.82,2


In [11]:
output_path = f'data/props/ev_analysis/underdog.csv'
df.to_csv(output_path, index=False)